## 准备数据

In [12]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [13]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [14]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))

        
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))

       
        ####################

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x_flattened = tf.reshape(x, [-1, 784])
        h1 = tf.matmul(x_flattened, self.W1) + self.b1
        activated_h1 = tf.tanh(h1)
        logits = tf.matmul(activated_h1, self.W2) + self.b2
        ####################
        return logits

model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [15]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [16]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.6595354 ; accuracy 0.068383336
epoch 1 : loss 2.6249194 ; accuracy 0.07571667
epoch 2 : loss 2.5919678 ; accuracy 0.082216665
epoch 3 : loss 2.5605152 ; accuracy 0.09
epoch 4 : loss 2.5304177 ; accuracy 0.09776667
epoch 5 : loss 2.50155 ; accuracy 0.10611667
epoch 6 : loss 2.4738026 ; accuracy 0.11473333
epoch 7 : loss 2.447077 ; accuracy 0.12368333
epoch 8 : loss 2.421287 ; accuracy 0.13245
epoch 9 : loss 2.3963537 ; accuracy 0.14075
epoch 10 : loss 2.3722088 ; accuracy 0.14888333
epoch 11 : loss 2.3487892 ; accuracy 0.15748334
epoch 12 : loss 2.3260393 ; accuracy 0.16655
epoch 13 : loss 2.303908 ; accuracy 0.17573333
epoch 14 : loss 2.2823505 ; accuracy 0.18456666
epoch 15 : loss 2.261326 ; accuracy 0.19335
epoch 16 : loss 2.2407973 ; accuracy 0.20188333
epoch 17 : loss 2.2207317 ; accuracy 0.21043333
epoch 18 : loss 2.2011 ; accuracy 0.21911667
epoch 19 : loss 2.1818748 ; accuracy 0.22796667
epoch 20 : loss 2.163032 ; accuracy 0.23671667
epoch 21 : loss 2.1445506 ; 